# Módulo 6 · Artefactos, canales malos e ICA

**Objetivo:** detectar tramos y canales contaminados, marcarlos para que MNE los ignore y
conocer ICA, la herramienta estrella de MNE para separar artefactos.

> ⚠️ **Aviso para tu TFM.** En detección/predicción de crisis con deep learning, lo habitual es un
> preprocesado **mínimo** (filtro + remuestreo) y dejar que la red aprenda a ignorar artefactos.
> Limpiar agresivamente (sobre todo con ICA) puede **borrar la propia crisis** (muchas crisis tienen
> componente muscular o de movimiento) y no se puede aplicar igual en tiempo real.
> Úsalo para **entender** los datos, para descartar ventanas imposibles (amplitudes absurdas,
> canales planos) y como experimento comparativo, no como paso obligatorio.

Artefactos típicos en TUSZ:
| Artefacto | Dónde | Aspecto |
|---|---|---|
| Parpadeo / ojos | FP1, FP2 | ondas lentas grandes, frontales |
| Músculo (EMG) | temporales | ruido rápido >20 Hz |
| Electrodo suelto | 1 canal | saltos, deriva, o señal plana |
| ECG | varios | picos regulares ~1 Hz |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, set_1020

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

raw = read_tusz_raw("aaaaabqv_s001_t000")
raw.pick("eeg").drop_channels(["A1", "A2"])     # en LE son la referencia (≈0)
set_1020(raw)                                  # descarta T1/T2 (sin posición estándar)
print(raw.ch_names)

## 6.1 Canales malos: `info["bads"]`

En el módulo 3 vimos que **OZ** deriva muchísimo. Comprobémoslo con números: desviación típica
robusta de cada canal comparada con la mediana.

In [ ]:
x = raw.get_data(units="uV")
mad = np.median(np.abs(x - np.median(x, 1, keepdims=True)), 1)
z = (mad - np.median(mad)) / (1.4826 * np.median(np.abs(mad - np.median(mad))))
pd.Series(z, index=raw.ch_names).sort_values(ascending=False).round(1).head(6)

Marcar un canal como malo **no lo borra**: lo apunta en `info["bads"]` y MNE lo excluye por defecto
de ICA, PSD, `pick("eeg")`... Se puede **interpolar** a partir de sus vecinos (necesita posiciones):

In [ ]:
raw.info["bads"] = ["OZ"]
print(raw.copy().pick("eeg").ch_names[-3:], "← OZ ya no aparece")

raw_interp = raw.copy().interpolate_bads(reset_bads=True)
fig, ax = plt.subplots(figsize=(12, 3))
t0, t1 = 200, 240
orig, tt = raw.get_data(picks="OZ", tmin=t0, tmax=t1, return_times=True, units="uV")
interp = raw_interp.get_data(picks="OZ", tmin=t0, tmax=t1, units="uV")
ax.plot(tt, orig[0], lw=0.6, label="OZ original")
ax.plot(tt, interp[0], lw=0.8, label="OZ interpolado (desde vecinos)")
ax.legend(); ax.set_xlabel("s"); ax.set_ylabel("µV"); plt.show()

## 6.2 Tramos malos automáticos: anotaciones `BAD_`

### Por amplitud pico a pico en ventanas (función propia)
El criterio más usado para descartar basura: si en una ventana de 1 s algún canal supera
X µV pico a pico, es casi seguro un artefacto (el EEG cerebral rara vez pasa de 300-500 µV).

In [ ]:
def annotate_ptp(raw, thresh_uv=800, win=1.0, desc="BAD_amp"):
    sf = raw.info["sfreq"]; n = int(win * sf)
    x = raw.get_data(picks="eeg", units="uV")
    n_win = x.shape[1] // n
    ptp = np.ptp(x[:, :n_win * n].reshape(x.shape[0], n_win, n), axis=2).max(0)   # máx. entre canales
    bad = np.where(ptp > thresh_uv)[0]
    return mne.Annotations(bad * win + raw.first_time, [win] * len(bad), [desc] * len(bad),
                           orig_time=raw.annotations.orig_time), ptp

raw_f = raw.copy().filter(0.5, 40)
ann_amp, ptp = annotate_ptp(raw_f, thresh_uv=800)
print(ann_amp, "→", ann_amp.duration.sum(), "s")
plt.figure(figsize=(12, 2.5)); plt.plot(ptp, lw=0.6); plt.axhline(800, c="r", ls="--")
plt.ylabel("µV pp (máx. canales)"); plt.xlabel("ventana de 1 s"); plt.show()

### Canales planos o con saltos: `annotate_amplitude`
La función de MNE `annotate_amplitude` mira la diferencia **entre muestras consecutivas**:
`flat` detecta tramos sin variación (electrodo desconectado) y `peak` saltos bruscos de una
muestra a la siguiente (no es un umbral pico a pico por ventana; para eso, la función de arriba).

In [ ]:
from mne.preprocessing import annotate_amplitude
ann_flat, bads_flat = annotate_amplitude(raw_f, flat=dict(eeg=1e-6), bad_percent=5, min_duration=0.1)
print("Canales planos (>5% del tiempo):", bads_flat)
print(ann_flat)

# ### Por músculo: `annotate_muscle_zscore`
# Filtra en alta frecuencia, calcula la envolvente y marca donde el z-score supera un umbral.
# El rango por defecto (110-140 Hz) no cabe en 250 Hz, así que lo bajamos a 30-100 Hz.

In [ ]:
from mne.preprocessing import annotate_muscle_zscore
ann_mus, scores = annotate_muscle_zscore(raw, ch_type="eeg", threshold=5, min_length_good=0.2,
                                         filter_freq=(30, 100))
print(ann_mus, "→", round(ann_mus.duration.sum(), 1), "s marcados")
plt.figure(figsize=(12, 2.5)); plt.plot(raw.times, scores, lw=0.5); plt.axhline(5, c="r", ls="--")
for a in raw.annotations:
    plt.axvspan(a["onset"], a["onset"] + a["duration"], color="tab:red", alpha=0.2)
plt.title("z-score de actividad muscular (rojo: crisis anotadas)"); plt.xlabel("s"); plt.show()

sz = [(a["onset"], a["onset"] + a["duration"]) for a in raw.annotations]
def overlap(ann, intervals):
    return sum(max(0, min(o + d, b) - max(o, a)) for o, d in zip(ann.onset, ann.duration) for a, b in intervals)
print(f"Segundos BAD_muscle que caen DENTRO de crisis: {overlap(ann_mus, sz):.1f} s de {sum(b - a for a, b in sz):.1f} s de crisis")

👀 **Los picos más altos del detector de músculo coinciden con las crisis.** La descarga punta-onda
tiene "puntas" con mucha energía en alta frecuencia. Si descartaras ciegamente los tramos `BAD_muscle`,
**perderías parte de las crisis**: justo lo que querías detectar. Moraleja: cualquier limpieza
automática hay que auditarla contra las anotaciones de crisis.

Combinamos las anotaciones (crisis + BAD) y las vemos:

In [ ]:
raw_f.set_annotations(raw.annotations + ann_amp + ann_flat + ann_mus)
pd.Series(raw_f.annotations.description).value_counts()

## 6.3 ICA: separar fuentes

ICA descompone los N canales en N (o menos) **componentes independientes**. Algunos serán
claramente artefactos (parpadeo, músculo, ECG) y se pueden quitar reconstruyendo la señal sin ellos.

Receta estándar:
1. Copia filtrada con paso alto **≥ 1 Hz** (ICA funciona mal con deriva lenta).
2. Referencia promedio.
3. `ICA(n_components=…).fit(...)`, excluyendo tramos `BAD_`.
4. Inspeccionar componentes, elegir cuáles quitar → `ica.exclude`.
5. `ica.apply()` sobre los datos que quieras limpiar.

In [ ]:
from mne.preprocessing import ICA

raw_ica = raw.copy().interpolate_bads().filter(1, 40).set_eeg_reference("average")
ica = ICA(n_components=15, method="fastica", random_state=42, max_iter="auto")
ica.fit(raw_ica, reject_by_annotation=True)
ica

In [ ]:
fig = ica.plot_components(show=False); plt.show()

In [ ]:
fig = ica.plot_sources(raw_ica, start=170, stop=200, show_scrollbars=False)

### Encontrar componentes oculares automáticamente
TUSZ no tiene canal EOG fiable, pero **FP1/FP2** hacen de proxy:

In [ ]:
eog_idx, eog_scores = ica.find_bads_eog(raw_ica, ch_name=["FP1", "FP2"], threshold=2.5)
print("Componentes oculares candidatos:", eog_idx)
fig = ica.plot_scores(eog_scores, show=False); plt.show()

### ¿Qué componente "es" la crisis?
Pregunta interesante: correlacionamos la potencia de cada componente con la máscara de crisis.

In [ ]:
src = ica.get_sources(raw_ica).get_data()
mask = np.zeros(raw_ica.n_times, bool)
for a in raw.annotations:
    mask[int(a["onset"] * 250):int((a["onset"] + a["duration"]) * 250)] = True
env = np.abs(src)
ratio = env[:, mask].mean(1) / env[:, ~mask].mean(1)
pd.Series(ratio, index=[f"ICA{i:03d}" for i in range(len(ratio))]).sort_values(ascending=False).round(2).head(5)

Si quitaras ese componente "por parecer un artefacto", **estarías borrando la crisis**. De ahí el aviso
del principio. Antes de excluir un componente ocular, comprueba también su ratio: si es alto, la
crisis "se ha colado" en él (la punta-onda es de predominio frontal, cerca de los ojos).

In [ ]:
print("Ratio crisis/fondo de los candidatos oculares:", {int(i): round(float(ratio[i]), 2) for i in eog_idx})
ica.exclude = list(map(int, eog_idx))
print("Excluimos:", ica.exclude)
raw_clean = ica.apply(raw_ica.copy())
fig = raw_clean.plot(start=170, duration=25, n_channels=19, scalings=dict(eeg=150e-6),
                     show_scrollbars=False, title="Tras ICA (componentes oculares eliminados)")

---
## 🧪 Ejercicios

**E6.1** Calcula el z-score robusto de canales (6.1) para `aaaaadqe_s001_t001`. ¿Tiene algún canal malo?

**E6.2** Cambia el umbral de `annotate_ptp` a 300, 500 y 1000 µV. ¿Cuántos segundos se
marcan como malos en cada caso? ¿Alguno de ellos cae **dentro de una crisis**? (Esto es justo lo que
no quieres al construir el dataset.)

**E6.3** Dibuja con `ica.plot_properties(raw_ica, picks=[...])` el componente con más ratio crisis/fondo.

### Solución E6.1

In [ ]:
def robust_z_channels(r):
    x = r.get_data(units="uV")
    mad = np.median(np.abs(x - np.median(x, 1, keepdims=True)), 1)
    return pd.Series((mad - np.median(mad)) / (1.4826 * np.median(np.abs(mad - np.median(mad)))),
                     index=r.ch_names)

r2 = read_tusz_raw("aaaaadqe_s001_t001").pick("eeg").drop_channels(["A1", "A2"])
robust_z_channels(r2).sort_values(ascending=False).round(1).head(5)

### Solución E6.2

In [ ]:
sz = [(a["onset"], a["onset"] + a["duration"]) for a in raw.annotations]
for pk in (300, 500, 1000):
    ann, _ = annotate_ptp(raw_f, thresh_uv=pk)
    print(f"{pk:5d} µV → {ann.duration.sum():6.1f} s malos, de ellos {overlap(ann, sz):5.1f} s dentro de crisis")

### Solución E6.3

In [ ]:
best = int(np.argmax(ratio))
figs = ica.plot_properties(raw_ica, picks=[best], show=False)
plt.show()